# myJourney: the map on my BIPM journey page

This notebook builds `map.html`, the interactive map embedded in [`index.html`](https://arslanchaudhry65.github.io/BIPM/).

It uses **folium**, a Python wrapper around the Leaflet map library, with the standard OpenStreetMap tiles. Those need **no API key**. Many other map services now do: Google Maps needs a key, Stadia (formerly Stamen) needs a registered account, and CARTO's free basemaps now draw "API key required" over every tile. That's why last year's embedded maps stopped working.

**How to use it:** run all cells. The last cell writes `map.html` next to the notebook; commit it together with `index.html`. In Google Colab, download `map.html` from the file panel on the left afterwards.

In [ ]:
# folium and pandas are preinstalled in Google Colab.
# Locally:  pip install folium pandas   (geopy only for the optional geocoding cell)
import html
import json

import folium
import pandas as pd
from branca.element import Element
from folium.features import DivIcon

## 1 · The stops

One row per station of my path, in chronological order. The row number becomes the marker number on the map and on the page.

The coordinates were geocoded once from the addresses (see the optional cell below) and are stored here, so the map builds offline and comes out the same every time.

In [ ]:
stops = pd.DataFrame(
    [
        # kind     place                       what                                                 period         area               address                                    lat       lon
        ("study", "Freie Universität Berlin", "Teacher training, Biology & Chemistry",             "2014 – 2020", "Dahlem",          "Takustraße 3, 14195 Berlin",              52.45472, 13.29641),
        ("study", "HTW Berlin",               "Bachelor, Business Administration",                 "2020 – 2024", "Karlshorst",      "Treskowallee 8, 10318 Berlin",            52.49320, 13.52462),
        ("work",  "Suxeedo",                  "Product Marketing Consultant",                      "2021 – 2022", "Mitte",           "Neue Schönhauser Straße 9, 10178 Berlin", 52.52478, 13.40384),
        ("work",  "PwC Deutschland",          "Product Owner",                                     "2022 – 2023", "Mitte",           "Kapelle-Ufer 4, 10117 Berlin",            52.52361, 13.37326),
        # Scheer IMC is a remote job, so this marker sits at my home station rather than at an office.
        ("work",  "Scheer IMC",               "Product Owner, remote",                             "since 2024",  "Prenzlauer Berg", "S Greifswalder Straße, Berlin",           52.54078, 13.43855),
        ("study", "HWR Berlin",               "M.Sc. Business Intelligence & Process Management",  "since 2026",  "Schöneberg",      "Badensche Straße 52, 10825 Berlin",       52.48546, 13.33820),
    ],
    columns=["kind", "place", "what", "period", "area", "address", "lat", "lon"],
)
stops.index = range(1, len(stops) + 1)  # 1-based, so it matches the marker numbers
stops

### Optional: look the coordinates up yourself

Set `GEOCODE = True` to geocode the addresses again with geopy and OpenStreetMap's Nominatim service. It needs internet access and waits about a second between requests, as Nominatim's usage policy asks.

In [ ]:
GEOCODE = False

if GEOCODE:
    from geopy.extra.rate_limiter import RateLimiter
    from geopy.geocoders import Nominatim

    geocode = RateLimiter(Nominatim(user_agent="bipm-journey-page").geocode, min_delay_seconds=1.1)
    found = stops["address"].apply(geocode)
    stops["lat"] = [loc.latitude for loc in found]
    stops["lon"] = [loc.longitude for loc in found]
    stops[["place", "lat", "lon"]]

## 2 · A small fact for the page

Every stop is in Berlin. How far apart are they? The haversine formula gives the great-circle distance between two coordinates.

In [ ]:
from itertools import combinations
from math import asin, cos, radians, sin, sqrt


def km(a, b):
    """Great-circle distance between two (lat, lon) points, in kilometres."""
    lat1, lon1, lat2, lon2 = map(radians, (*a, *b))
    h = sin((lat2 - lat1) / 2) ** 2 + cos(lat1) * cos(lat2) * sin((lon2 - lon1) / 2) ** 2
    return 2 * 6371 * asin(sqrt(h))


points = stops[["lat", "lon"]].values.tolist()
path_km = sum(km(a, b) for a, b in zip(points, points[1:]))
i, j = max(combinations(range(len(points)), 2), key=lambda p: km(points[p[0]], points[p[1]]))

print(f"Path, stop to stop:  {path_km:.1f} km")
print(f"Farthest apart:      {stops.place.iloc[i]} and {stops.place.iloc[j]}, {km(points[i], points[j]):.1f} km")

## 3 · Build the map

The look follows the page: warm paper tones, one petrol accent, IBM Plex type and square numbered markers. Two details make it feel like part of the page rather than a pasted-in widget:

- **Colour.** OpenStreetMap's own style is colourful. A CSS filter turns the tiles grey and warms them to paper tones; in dark mode the same filter also inverts them.
- **Theme.** The map starts in the reader's light or dark preference. When the theme toggle on the page is pressed, the page sends the map a message and the map switches with it.
- **Stops list.** Selecting a stop in the list next to the map sends another message; the map flies to that marker and opens its popup.

In [ ]:
# "OpenStreetMap" is folium's built-in name for the standard OSM tiles, attribution included.
# scroll_wheel_zoom=False: scrolling the page should scroll the page, not zoom the map.
m = folium.Map(tiles="OpenStreetMap", scroll_wheel_zoom=False)

# The path, drawn in the order the stops happened.
folium.PolyLine(points, weight=2, opacity=0.9, dash_array="2 6", class_name="journey-line").add_to(m)

markers = []
for n, stop in stops.iterrows():
    is_now = n == stops.index[-1]
    icon = DivIcon(
        html=f'<div class="stop-marker{" is-now" if is_now else ""}">{n:02d}</div>',
        icon_size=(24, 24),
        icon_anchor=(12, 12),
        class_name="stop-icon",
    )
    popup = folium.Popup(
        '<div class="pop">'
        f'<div class="pop-meta">{html.escape(stop.period)}</div>'
        f'<div class="pop-title">{html.escape(stop.place)}</div>'
        f'<div class="pop-text">{html.escape(stop.what)}</div>'
        f'<div class="pop-area">{html.escape(stop.area)}</div>'
        "</div>",
        max_width=260,
    )
    tooltip = folium.Tooltip(html.escape(stop.place), direction="top", offset=(0, -14))
    marker = folium.Marker((stop.lat, stop.lon), icon=icon, popup=popup, tooltip=tooltip).add_to(m)
    markers.append(marker)

m.fit_bounds(points, padding=(40, 40))

### Styling and behaviour

The CSS carries a small subset of my design tokens for light and dark. The JavaScript keeps all six stops in frame and listens for the page's messages.

In [ ]:
STYLE = """
<link rel="stylesheet" href="assets/plex.css">
<style>
  :root {
    --ac-paper-0: #FDFBF7; --ac-paper-1: #FBF8F3; --ac-paper-2: #F7F3EC; --ac-paper-3: #F0EAE0;
    --ac-line: #E7E0D5; --ac-line-strong: #D8CFC0;
    --ac-ink-0: #1F1B15; --ac-ink-1: #3D382F; --ac-ink-2: #5E5649; --ac-ink-3: #756D62;
    --ac-accent: #135E63; --ac-accent-hover: #0F4E52; --ac-on-accent: #FBF8F3;
    --ac-shadow-popover: 0 6px 20px rgba(31, 27, 21, 0.10), 0 1px 3px rgba(31, 27, 21, 0.06);
    --ac-font-sans: 'IBM Plex Sans', -apple-system, BlinkMacSystemFont, 'Segoe UI', Helvetica, Arial, sans-serif;
    --ac-font-mono: 'IBM Plex Mono', ui-monospace, 'SF Mono', Menlo, Consolas, monospace;
    --tile-filter: grayscale(1) sepia(0.2) contrast(0.72) brightness(1.16);
  }
  :root[data-theme="dark"] {
    --ac-paper-0: #221E18; --ac-paper-1: #16130F; --ac-paper-2: #100E0B; --ac-paper-3: #2B2620;
    --ac-line: #322C23; --ac-line-strong: #423B2F;
    --ac-ink-0: #EDE6D9; --ac-ink-1: #C5BAA8; --ac-ink-2: #A79C8C; --ac-ink-3: #908678;
    --ac-accent: #4FB8BE; --ac-accent-hover: #6FC9CE; --ac-on-accent: #0F2E30;
    --ac-shadow-popover: 0 6px 20px rgba(0, 0, 0, 0.45), 0 1px 3px rgba(0, 0, 0, 0.30);
    --tile-filter: grayscale(1) invert(1) sepia(0.3) brightness(0.8) contrast(0.9);
  }

  html, body, .leaflet-container { background: var(--ac-paper-2); }
  .leaflet-container { font-family: var(--ac-font-sans); color: var(--ac-ink-1); }
  .leaflet-container a { color: var(--ac-accent); }
  .leaflet-tile-pane { filter: var(--tile-filter); }   /* OSM colours -> warm greys */

  .journey-line { stroke: var(--ac-accent); }

  .stop-icon { background: none; border: 0; }
  .stop-marker {
    box-sizing: border-box; width: 24px; height: 24px;
    display: grid; place-items: center;
    font-family: var(--ac-font-mono); font-size: 11px; font-weight: 500; line-height: 1;
    color: var(--ac-accent); background: var(--ac-paper-0);
    border: 1px solid var(--ac-accent); border-radius: 2px;
  }
  .stop-marker.is-now { background: var(--ac-accent); color: var(--ac-on-accent); }

  .leaflet-popup-content-wrapper {
    background: var(--ac-paper-0); color: var(--ac-ink-1);
    border: 1px solid var(--ac-line-strong); border-radius: 4px;
    box-shadow: var(--ac-shadow-popover);
  }
  .leaflet-popup-tip { background: var(--ac-paper-0); border: 1px solid var(--ac-line-strong); box-shadow: none; }
  .leaflet-popup-content { margin: 12px 16px; font-size: 13px; line-height: 1.5; }
  .leaflet-container a.leaflet-popup-close-button { color: var(--ac-ink-3); }
  .leaflet-container a.leaflet-popup-close-button:hover { color: var(--ac-ink-0); }
  .pop-meta {
    font-family: var(--ac-font-mono); font-size: 11px; text-transform: uppercase;
    letter-spacing: 0.05em; color: var(--ac-accent);
  }
  .pop-title { margin-top: 4px; font-size: 14px; font-weight: 500; color: var(--ac-ink-0); }
  .pop-text  { color: var(--ac-ink-2); }
  .pop-area  { margin-top: 4px; font-family: var(--ac-font-mono); font-size: 11px; color: var(--ac-ink-3); }

  .leaflet-tooltip {
    background: var(--ac-paper-0); color: var(--ac-ink-0);
    border: 1px solid var(--ac-line-strong); border-radius: 2px; box-shadow: none;
    font-family: var(--ac-font-sans); font-size: 12px; padding: 4px 8px;
  }
  .leaflet-tooltip-top::before    { border-top-color: var(--ac-line-strong); }
  .leaflet-tooltip-bottom::before { border-bottom-color: var(--ac-line-strong); }
  .leaflet-tooltip-left::before   { border-left-color: var(--ac-line-strong); }
  .leaflet-tooltip-right::before  { border-right-color: var(--ac-line-strong); }

  .leaflet-bar, .leaflet-touch .leaflet-bar {
    border: 1px solid var(--ac-line-strong); border-radius: 4px; box-shadow: none;
  }
  .leaflet-bar a, .leaflet-bar a:hover {
    background: var(--ac-paper-0); color: var(--ac-ink-1); border-bottom-color: var(--ac-line);
  }
  .leaflet-bar a:hover { background: var(--ac-paper-3); }
  .leaflet-bar a.leaflet-disabled { color: var(--ac-ink-3); background: var(--ac-paper-2); }
  .leaflet-container .leaflet-control-attribution {
    background: var(--ac-paper-0); color: var(--ac-ink-3); font-size: 11px;
  }
  :focus-visible { outline: 2px solid var(--ac-accent); outline-offset: 2px; }
</style>
"""

SCRIPT = """
document.addEventListener('DOMContentLoaded', function () {
  var map = __MAP__;
  var markers = [__MARKERS__];
  var bounds = L.latLngBounds(__POINTS__);
  var root = document.documentElement;
  var reduceMotion = window.matchMedia('(prefers-reduced-motion: reduce)').matches;
  var framed = true;   // true while the map still shows the whole path

  function fitAll(animate) {
    framed = true;
    map.closePopup();
    if (animate && !reduceMotion) map.flyToBounds(bounds, { padding: [40, 40], duration: 0.8 });
    else map.fitBounds(bounds, { padding: [40, 40] });
  }

  function show(marker) {
    framed = false;
    var zoom = Math.max(map.getZoom(), 14);
    map.closePopup();
    map.once('moveend', function () { marker.openPopup(); });
    if (reduceMotion) map.setView(marker.getLatLng(), zoom);
    else map.flyTo(marker.getLatLng(), zoom, { duration: 0.8 });
  }

  function setTheme(mode) { root.dataset.theme = mode; }

  setTheme(window.matchMedia('(prefers-color-scheme: dark)').matches ? 'dark' : 'light');

  // The page may load this map while it is still hidden or tiny; re-frame once it has its real size.
  map.on('dragstart', function () { framed = false; });
  window.addEventListener('resize', function () {
    map.invalidateSize();
    if (framed) fitAll(false);
  });
  window.addEventListener('load', function () { map.invalidateSize(); fitAll(false); });

  // Messages from the page that embeds this map.
  window.addEventListener('message', function (event) {
    if (event.source !== window.parent) return;
    var msg = event.data || {};
    if (msg.type === 'theme' && (msg.mode === 'light' || msg.mode === 'dark')) setTheme(msg.mode);
    if (msg.type === 'focus' && Number.isInteger(msg.index) && markers[msg.index]) show(markers[msg.index]);
    if (msg.type === 'reset') fitAll(true);
  });
});
"""

script = (
    SCRIPT.replace("__MAP__", m.get_name())
    .replace("__MARKERS__", ", ".join(mk.get_name() for mk in markers))
    .replace("__POINTS__", json.dumps(points))
)

# Into the body rather than the head: folium puts Leaflet's own stylesheet into the head at save time,
# and this CSS has to come after it to win.
m.get_root().html.add_child(Element(STYLE))
m.get_root().script.add_child(Element(script))

## 4 · Save

`map.html` is self-contained apart from the map library and tiles it loads from the web, plus `assets/plex.css` for the fonts. It has to sit in the same folder as `index.html`.

In [ ]:
m.save("map.html")
print("map.html written")
m